# 🏆 Master Benchmark Comparison & Publication Visualizations
### **Jadavpur University Research Internship | Under: Prof. Tohida Rehman**
**Track:** FIRE MixSub-SciHigh Scientific Highlight Summarization

---
## 📌 Strict Scientific Benchmark Setup
- **Identical Token Limits Across All Models**: `MAX_SOURCE_LENGTH = 512` & `MAX_TARGET_LENGTH = 128`
- **The 3 Non-Redundant Core Research Pillars Evaluated:**

| Pillar | Core Metric | What It Means | Direction (Better) |
| :--- | :--- | :--- | :---: |
| **Pillar 1: Overall Quality** | **ROUGE-L F1** & **Target F1** | Syntactic fluency and semantic overlap with ground-truth author highlights | **Higher is Better (↑)** |
| **Pillar 2: Hallucination Risk** | **Ungrounded Token %** & **Source Grounding %** | Percentage of claims made that have ZERO basis in the source paper abstract | **Lower is Better (↓)** |
| **Pillar 3: Inference Time** | **Wall-Clock Latency (sec)** & **Throughput (tok/s)** | Real-world GPU execution time per paper and generation serving speed | **Lower Latency (↓) / Higher Tok/s (↑)** |


In [ ]:
# Step 1: Mount Google Drive
from google.colab import drive
import os

drive.mount('/content/drive')
DRIVE_BASE_DIR = '/content/drive/MyDrive/JU_Benchmark_Project'
RESULTS_DIR = os.path.join(DRIVE_BASE_DIR, 'results')
PLOTS_DIR = os.path.join(RESULTS_DIR, 'plots')
os.makedirs(PLOTS_DIR, exist_ok=True)

print('📁 Reading results from:', RESULTS_DIR)
print('📁 Saving plots to:     ', PLOTS_DIR)


In [ ]:
# Step 2: Install Libraries
!pip install -q tabulate matplotlib seaborn pandas


In [ ]:
# Step 3: Compile and Display 3-Pillar Master Benchmark Table
import glob
import json
import pandas as pd
from tabulate import tabulate

json_files = glob.glob(os.path.join(RESULTS_DIR, 'benchmark_*.json'))

if not json_files:
    print('⚠️ No benchmark_*.json files found yet in Drive results folder.')
    print('Please run individual model notebooks (01 to 06) first!')
else:
    all_metrics = []
    for jf in sorted(json_files):
        with open(jf, 'r') as f:
            all_metrics.append(json.load(f))

    summary_rows = []
    for m in all_metrics:
        summary_rows.append({
            'Model': m.get('model', 'N/A').upper(),
            # Pillar 1: Overall Quality (↑)
            'Overall Quality [ROUGE-L F1] (↑)': f"{m.get('overall_quality_rougeL_f1', 0)*100:.2f}%",
            'Target F1 Score (↑)': f"{m.get('f1_score_target', 0)*100:.2f}%",
            # Pillar 2: Hallucination Risk (↓)
            'Hallucination Risk [% Ungrounded] (↓)': f"{m.get('hallucination_risk_pct', 0)*100:.2f}%",
            'Source Grounding [% Grounded] (↑)': f"{m.get('source_grounding_precision', 0)*100:.2f}%",
            # Pillar 3: Inference Time (↓/↑)
            'Wall-Clock Latency (↓)': f"{m.get('wall_clock_latency_sec', 0):.2f}s",
            'Throughput (↑)': f"{m.get('throughput_tokens_per_sec', 0):.1f} tok/s"
        })

    comparison_df = pd.DataFrame(summary_rows)
    master_csv = os.path.join(RESULTS_DIR, 'master_3pillar_benchmark_comparison.csv')
    comparison_df.to_csv(master_csv, index=False)

    print('\n' + '='*100)
    print('🏆 3-PILLAR SCIENTIFIC BENCHMARK MATRIX (Quality vs Hallucination vs Latency)')
    print('='*100)
    print(tabulate(comparison_df, headers='keys', tablefmt='github', showindex=False))
    print(f'\n📁 Master CSV saved to: {master_csv}')


In [ ]:
# Step 4: Render High-Resolution 300-DPI Visual Figures (The 3 Pillars)
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

if not json_files:
    print('Cannot plot: No benchmark results found.')
else:
    df_raw = pd.DataFrame(all_metrics)
    sns.set_theme(style='whitegrid', font_scale=1.1)
    PALETTE = ['#2b5c8f', '#e26d5c', '#38b000', '#7209b7', '#f77f00', '#4cc9f0']

    # Figure 1: Pillar 1 vs Pillar 3 -> Overall Quality vs. Inference Time (Pareto Frontier)
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    x = df_raw['wall_clock_latency_sec']
    y = df_raw['overall_quality_rougeL_f1'] * 100
    ax.scatter(x, y, c=range(len(df_raw)), cmap='viridis', s=500, alpha=0.9, edgecolors='black', linewidth=1.5, zorder=3)
    for i, txt in enumerate(df_raw['model']):
        ax.annotate(f' {txt.upper()}', (x.iloc[i], y.iloc[i]), fontsize=10, fontweight='bold', xytext=(8, 4), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.25', facecolor='#ffffff', edgecolor='#bbbbbb', alpha=0.9))
    ax.set_title('Pillar 1 vs. 3: Overall Quality (ROUGE-L F1 ↑) vs. Inference Time (Latency ↓)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Wall-Clock Latency per Sample (seconds) → Lower is Faster (↓)', fontsize=11, labelpad=10)
    ax.set_ylabel('Overall Quality ROUGE-L F1 (%) → Higher is Better (↑)', fontsize=11, labelpad=10)
    ax.axvspan(0, x.median(), color='#e8f5e9', alpha=0.35, label='High Efficiency Zone', zorder=1)
    ax.legend(loc='lower right', frameon=True)
    plt.tight_layout()
    f1 = os.path.join(PLOTS_DIR, 'quality_vs_inference_latency.png')
    plt.savefig(f1, bbox_inches='tight')
    plt.show()
    print('Saved:', f1)

    # Figure 2: Pillar 1 vs Pillar 2 -> Overall Quality vs. Hallucination Risk
    fig, ax = plt.subplots(figsize=(10, 6), dpi=300)
    x_hal = df_raw['hallucination_risk_pct'] * 100
    y_qual = df_raw['overall_quality_rougeL_f1'] * 100
    ax.scatter(x_hal, y_qual, c=range(len(df_raw)), cmap='coolwarm', s=500, alpha=0.9, edgecolors='black', linewidth=1.5, zorder=3)
    for i, txt in enumerate(df_raw['model']):
        ax.annotate(f' {txt.upper()}', (x_hal.iloc[i], y_qual.iloc[i]), fontsize=10, fontweight='bold', xytext=(8, 4), textcoords='offset points',
                    bbox=dict(boxstyle='round,pad=0.25', facecolor='#ffffff', edgecolor='#bbbbbb', alpha=0.9))
    ax.set_title('Pillar 1 vs. 2: Overall Quality (ROUGE-L F1 ↑) vs. Hallucination Risk (↓)', fontsize=14, fontweight='bold', pad=15)
    ax.set_xlabel('Hallucination Risk (% Ungrounded in Source) → Lower is Safer (↓)', fontsize=11, labelpad=10)
    ax.set_ylabel('Overall Quality ROUGE-L F1 (%) → Higher is Better (↑)', fontsize=11, labelpad=10)
    ax.axvspan(0, x_hal.median(), color='#e3f2fd', alpha=0.35, label='Safe Grounding Zone', zorder=1)
    ax.legend(loc='lower left', frameon=True)
    plt.tight_layout()
    f2 = os.path.join(PLOTS_DIR, 'quality_vs_hallucination_risk.png')
    plt.savefig(f2, bbox_inches='tight')
    plt.show()
    print('Saved:', f2)

    # Figure 3: Hallucination Risk Breakdown Across Models
    fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
    models = [m.upper() for m in df_raw['model']]
    bar_colors = ['#2a9d8f' if v < df_raw['hallucination_risk_pct'].median() else '#e76f51' for v in df_raw['hallucination_risk_pct']]
    ax.bar(models, df_raw['hallucination_risk_pct'] * 100, color=bar_colors, edgecolor='black', linewidth=0.6)
    ax.axhline(df_raw['hallucination_risk_pct'].mean() * 100, color='red', linestyle='--', label=f"Average Risk ({df_raw['hallucination_risk_pct'].mean()*100:.1f}%)")
    ax.set_title('Pillar 2: Hallucination Risk (% of Claims Ungrounded in Source Abstract ↓)', fontsize=13, fontweight='bold', pad=15)
    ax.set_ylabel('Ungrounded Token % (Lower is Safer ↓)', fontsize=11)
    ax.tick_params(axis='x', rotation=20)
    ax.legend(frameon=True)
    plt.tight_layout()
    f3 = os.path.join(PLOTS_DIR, 'hallucination_risk_ranking.png')
    plt.savefig(f3, bbox_inches='tight')
    plt.show()
    print('Saved:', f3)

    # Figure 4: Inference Throughput (Tokens/sec) & Latency Dual Panel
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5), dpi=300)
    ax1.bar(models, df_raw['wall_clock_latency_sec'], color='#f4a261', edgecolor='black', linewidth=0.6)
    ax1.set_title('Pillar 3: Wall-Clock Latency (Seconds/Sample ↓)', fontsize=12, fontweight='bold')
    ax1.set_ylabel('Seconds (Lower is Faster ↓)', fontsize=10)
    ax1.tick_params(axis='x', rotation=25)

    if 'throughput_tokens_per_sec' in df_raw.columns:
        ax2.bar(models, df_raw['throughput_tokens_per_sec'], color='#2a9d8f', edgecolor='black', linewidth=0.6)
        ax2.set_title('Pillar 3: Generation Throughput (Tokens/Sec ↑)', fontsize=12, fontweight='bold')
        ax2.set_ylabel('Tokens per Second (Higher is Better ↑)', fontsize=10)
        ax2.tick_params(axis='x', rotation=25)

    plt.tight_layout()
    f4 = os.path.join(PLOTS_DIR, 'inference_speed_latency.png')
    plt.savefig(f4, bbox_inches='tight')
    plt.show()
    print('Saved:', f4)

    print('\n🎉 All 4 non-redundant research figures generated and saved to Drive!')
